In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import os
from natsort import natsorted

import scanpy as sc
import seaborn as sns

from scroutines import basicu

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning
from tqdm import tqdm


import sys
sys.path.insert(0, '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/myvisctx/analysis_multiome/')
import lmm

In [2]:
%%time
outfigdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/'
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_multiome_IT.h5ad'
adata_raw = sc.read(f)
adata_raw

CPU times: user 854 ms, sys: 8.96 s, total: 9.82 s
Wall time: 10.9 s


AnnData object with n_obs × n_vars = 89287 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time'
    var: 'feature_types'
    layers: 'norm'

In [3]:
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/L6IT_labels_gao25_to_yoo25_knn.csv' 
df_lbl = pd.read_csv(f)
df_lbl

,label,conf
AAACCGAAGGAGTCGG-1-P21a-2023 Multiome-9-0,37_L6 IT CTX Glut_1,0.347887
AAACGCGCAAACTGTT-1-P21a-2023 Multiome-9-0,51_L6 IT CTX Glut_4,1.000000
AAAGCGGGTCTTTGAC-1-P21a-2023 Multiome-9-0,52_L6 IT CTX Glut_4,0.677922
AACAGATAGCAAGATG-1-P21a-2023 Multiome-9-0,37_L6 IT CTX Glut_1,0.872321
AACAGATAGTTTGAGC-1-P21a-2023 Multiome-9-0,50_L6 IT CTX Glut_4,0.606479
...,...,...
TTAGGCTAGCCTAACG-1-P21DRa-2023 Multiome-10-0,50_L6 IT CTX Glut_4,0.826069
CTTGTTTAGGAGCAAC-1-P21DRa-2023 Multiome-10-0,37_L6 IT CTX Glut_1,0.563912
AAATGCCTCACAGCGC-1-P21DRb-2023 Multiome-10-0,41_L6 IT CTX Glut_2,0.536286
GATTATGTCTAAATCG-1-P21DRa-2023 Multiome-10-0,37_L6 IT CTX Glut_1,0.807793


In [4]:
adata = adata_raw[df_lbl.index].copy()
adata.obs = adata.obs.join(df_lbl)
adata.obs
adata

AnnData object with n_obs × n_vars = 1718 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time', 'label', 'conf'
    var: 'feature_types'
    layers: 'norm'

In [5]:
adata.X.data

array([27.,  1.,  2., ...,  4.,  4.,  8.], dtype=float32)

In [6]:
adata.obs['Age'].unique()

['P21', 'P21DR']
Categories (2, object): ['P21', 'P21DR']

In [7]:
adata.obs['Sample'].unique()

['P21a', 'P21b', 'P21DRb', 'P21DRa']
Categories (4, object): ['P21DRa', 'P21DRb', 'P21a', 'P21b']

In [8]:
adata.obs['total_counts'].unique()

array([12991., 24247., 11462., ..., 11217.,  7121., 15104.], dtype=float32)

In [9]:
clusters = np.sort(adata.obs['label'].unique())
clusters

array(['37_L6 IT CTX Glut_1', '41_L6 IT CTX Glut_2',
       '50_L6 IT CTX Glut_4', '51_L6 IT CTX Glut_4',
       '52_L6 IT CTX Glut_4'], dtype=object)

In [10]:
import time

In [11]:
%%time

obs_fixed1 = 'Age'
obs_fixed2 = None # 'Light'
obs_random = 'Sample'

cluster_col = 'label'

offset = 1e-2
scale = 1e4

for cluster in clusters:
    tag = f"d260303_{cluster.replace('/', '').replace(' ', '_')}"
    output = os.path.join(outfigdir, f'NRDR_DEGs_LMM_yoo25_P21_{tag}.csv')

    adatasub = adata[adata.obs[cluster_col]==cluster]
    genes = adatasub.var.index.values 

    if obs_fixed2 is None:
        obs = adatasub.obs[[obs_fixed1, obs_random]].copy()
    else:
        obs = adatasub.obs[[obs_fixed1, obs_fixed2, obs_random]].copy()
    obs = obs.dropna()
    adatasub = adatasub[obs.index]

    # mat_raw = np.array(adatasub.X.todense())
    # mat_raw = np.array(adatasub.raw.X.todense())
    mat_raw = np.array(adatasub.X.todense())
    
    # ### test
    # adatasub = adatasub[:,:20]
    # genes = genes[:20]
    # mat_raw = mat_raw[:,:20]
    # ### test

    # mat (CP10k norm)
    # mat = mat_raw/adatasub.obs['n_counts'].values.reshape(-1,1)*scale
    mat = mat_raw/adatasub.obs['total_counts'].values.reshape(-1,1)*scale

    res = lmm.run_lmm(mat, genes, obs, obs_fixed1, obs_random, output_csv=output, offset=offset)
    print(output)

(505, 16567) (505, 2)
(505, 15445) (505, 2)
(505, 9576) (505, 2)
463 ['Alkal1' 'Dnah7c' 'Cflar' 'March4' 'Cfap65' 'Dnajb2' 'Cops9' 'Gpc1'
 'Slco4c1' 'Cdh20' 'Nckap5' 'Tmem163' 'Rassf5' 'Slc41a1' 'Cdk18' 'Pik3c2b'
 'Etnk2' 'Btg2' 'Tmem9' 'Colgalt2' 'Apobec4' 'Lamc1' 'Ier5' 'Gm2000'
 'Soat1' 'Tor3a' 'Gm16701' 'Usf1' 'Atp1a2' 'Opn3' 'Sccpdh' 'Stum'
 'Gm36388' 'Kcnk2' 'Cd34' '1700080N15Rik' 'Pter' 'Anapc2' 'Ptgds' 'Ier5l'
 'St6galnac6' 'Angptl2' 'Gsn' 'Gm13481' 'Rbms1' 'Tbr1' 'Gm13561' 'Scrn3'
 'Rtn4rl2' 'Trp53i11' 'Gm13963' 'Gm13974' 'Gpr176' 'Bahd1' 'Rmdn3' 'Sord'
 'Gatm' 'Mal' 'Ebf4' 'Cdc25b' '4930545L23Rik' 'Lamp5' 'Cfap61' 'Zcchc3'
 'Nnat' 'Ctsa' 'Cdh22' 'Kcng1' 'Gm20721' 'Cdh4' 'Hrh3' 'Adrm1' 'Ntsr1'
 'Arfrp1' 'Clcn5' 'Sept6' 'Tmem255a' 'Enox2' 'Nsdhl' 'Emd' 'Prrg1'
 'Tmem47' 'Xist' 'Pcdh11x' 'Rpl36a' 'Nhs' 'Prps2' 'Arhgap6' 'Gm15246'
 'Trpc3' 'Lhfp' 'P2ry14' 'A730090N16Rik' 'Tiparp' 'Gba' 'Nup210l' 'Cgn'
 'Ctss' 'Ciart' 'Lix1l' 'Gm15886' 'Slc16a1' 'Ampd2' 'Amigo1' 'Col11a1'
 'Olfm3'

100% 9576/9576 [13:39<00:00, 11.69it/s]


97 ['March4' 'Rassf5' 'Pik3c2b' 'Lamc1' 'Stum' 'Cd34' 'Anapc2' 'Rtn4rl2'
 'Trp53i11' 'Bahd1' '4930545L23Rik' 'Nsdhl' 'Emd' 'Xist' 'Prps2' 'Lhfp'
 'Nup210l' 'Lix1l' 'Gm15886' 'Amigo1' 'Olfm3' 'Sec24d' 'Lpar1' 'Ror1'
 'Ak4' 'Ssbp3' 'Podn' 'Foxo6' 'Stk40' 'Map7d1' 'Serinc2' 'Lypla2' 'Ephb2'
 'Wnt4' 'Tnfrsf25' 'Cdk6' 'Agbl5' 'Sel1l3' 'Parm1' 'Sgsm1' 'Orai1'
 'Plxna1' 'Prkd2' 'Dpf1' 'Sipa1l3' 'A730056A06Rik' 'Gm14372' 'Tpd52l1'
 'Egr2' 'Chst11' 'Cotl1' 'Cbfa2t3' '2810455O05Rik' 'Kctd6' 'Med4' 'Pde4a'
 'Nectin1' 'Sik2' 'Cyp11a1' 'Car12' 'Trim71' 'Kremen1' 'Gm12198' 'Per1'
 'Efnb3' 'Tsr1' '1700016P03Rik' 'Rhbdl3' 'Cacnb1' 'Stac2' 'Rara' 'Gm11655'
 'Hid1' 'Faap100' 'Fst' 'Gm48342' 'Trib2' 'Odc1' 'Otub2' 'Tunar' 'Zhx2'
 'Myh9' 'Ccdc134' 'C730034F03Rik' 'Lmbr1l' 'Asic1' 'Grasp' 'Mrps6' 'Synj2'
 'Gnptg' 'Sox8' 'Dusp1' 'Rftn1' 'Sh3gl1' 'Eif2s3y' 'Slc6a7' 'Cstf2t']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_37_L6_IT_CTX_Glut_1.csv
/u/ho

100% 9512/9512 [13:58<00:00, 11.35it/s]


97 ['Cflar' 'Nhej1' 'Etnk2' 'Colgalt2' 'Apobec4' 'Mgst3' 'Usp21' 'Lin9'
 'Tgfb2' 'Pfkfb3' 'Xkr7' 'Cdh22' 'Ddx27' 'Xist' 'Gm6209' 'Tiparp'
 'Gm17501' 'Spata1' 'Gm12394' 'Gm12536' 'Lpar1' 'Susd1' 'Whrn' 'Lrp8os2'
 'Plk3' 'Foxo6' 'Serinc2' 'Selenon' 'Lypla2' 'Ephb2' 'Disp3' 'Tnfrsf25'
 'Prkg2' 'Sgsm1' 'Tmem120a' 'Gadd45a' 'Ttll3' 'Foxj2' 'Zscan22' 'Prkd2'
 'Cxcl17' 'Gm15413' 'Gm15398' 'Irs2' 'Mast3' 'Nutf2' 'Cbfa2t3' 'Arv1'
 'Sh2d4b' '4930579G18Rik' 'Egr3' 'Gm48293' 'Ubash3b' 'Sik2' 'Gm17231'
 'Arid3b' 'Megf11' 'Ints14' 'Car12' 'Jade2' 'Kdm6b' '1700016P03Rik'
 'Stac2' 'Rara' 'Map3k14' 'Sstr2' 'Sox4' 'Cage1' 'Gm34354' 'Golm1'
 'Polr3g' 'Homer1' 'Sv2c' 'Gm48342' 'Trib2' 'Fam71d' 'Fam161b' 'Tmem196'
 'Trib1' 'Pdgfb' 'Creld2' 'Igfbp6' 'Pkp2' '2510009E07Rik' 'Arhgap31'
 'Synj2' '4930506C21Rik' 'Sox8' 'Runx2os1' 'Foxn2' '4930480K15Rik' 'Kdm5d'
 'Eif2s3y' 'Uty' 'Ddx3y' 'Lipg' 'Mamdc2']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_41_L6

...

41 ['Gm1992' 'Kif26b' 'Stum' 'Gtf3c4' 'Gca' 'Rbm45' 'Pim2' 'Xist' 'Nhlrc3'
 'Fpgt' 'Pdp1' 'Bag1' 'Amz1' 'Zfp12' 'Cav1' 'Ggcx' 'Ybx3' 'Erf' 'Polr2i'
 'Vip' 'Thop1' 'Krr1' 'Arhgef25' 'Rab8a' 'Smad1' 'Tmem170' '2310009A05Rik'
 'Mpg' 'Mrpl55' 'Cox10' 'Ube2o' 'Itpk1' 'Syne3' 'Klf10' 'Pdzrn4' 'Ppm1f'
 'Mrpl40' 'Aars2' 'Cul9' 'Kctd1' '1700086O06Rik']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_50_L6_IT_CTX_Glut_4.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_50_L6_IT_CTX_Glut_4.csv
(381, 16567) (381, 2)
(381, 15514) (381, 2)
(381, 9672) (381, 2)
389 ['Imp4' 'Mrpl30' 'Pdcl3' 'Gm28055' 'Nrp2' 'Zdbf2' 'Rpl37a' 'Stk11ip'
 'Sgpp2' 'Htr2b' 'Kcnj13' 'Sag' 'Per2' 'Cdh7' 'Dbi' 'Rassf5' 'Snrpe' 'Npl'
 'Gas5' 'Lin9' 'Gm37768' 'Traf5' 'Camk1g' 'Pfkfb3' 'Arl5b' 'Bmi1'
 'Gm17171' 'Asb6' 'Ak1' 'Mrrf' 'Rnd3' 'Tnfaip6' 'Scrn3' 'Slc35c1' 'Trim69'
 'Lzts3' 'Lamp5' 'Banf2' 'Cst3' 'Rbck1' 'Tti1' 'Rims4

100% 9672/9672 [08:45<00:00, 18.40it/s]


62 ['Zdbf2' 'Per2' 'Rassf5' 'Camk1g' 'Pfkfb3' 'Arl5b' 'Lzts3' 'Rims4' 'Bcor'
 'Mamld1' 'Xist' 'Tiparp' '4921511C10Rik' 'Rps3a1' 'Ror1' 'Pla2g5' 'Rcc2'
 'Disp3' 'Galnt9' 'Mest' 'Ppm1k' 'Grip2' 'Tmem91' 'Numbl' 'Ppfibp2'
 'Gsg1l' 'Sbk1' 'Tnfrsf23' 'Lrrc20' 'Rsph14' 'Mast3' 'Ddx39' 'Cenpn'
 'Cbfa2t3' 'Sh2d4b' 'Ppp2r1b' 'Sik2' 'Layn' 'Gm17231' 'Smad3'
 '4930422M22Rik' 'Gm12064' 'Guk1' 'Mis12' 'Homer1' 'Trib2' 'Tunar'
 'Gm26854' 'Pla2g6' 'Csdc2' 'Phf21b' 'Btg3' 'Sox8' 'Cdkn1a' 'Zfp36l2'
 'Plekhh2' 'Kdm5d' 'Eif2s3y' 'Smad7' 'Pqlc1' 'Pcx' 'Wbp1l']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_51_L6_IT_CTX_Glut_4.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_51_L6_IT_CTX_Glut_4.csv
(166, 16567) (166, 2)
(166, 14548) (166, 2)
(166, 10075) (166, 2)
2150 ['Gm26901' 'Adhfe1' 'Mcmdc2' ... 'mt-Nd6' 'Vamp7' 'CAAA01118383.1']


100% 10075/10075 [09:17<00:00, 18.07it/s]


46 ['Map4k4' 'Tpp2' 'Ndufa10' 'Rgl1' 'Gm37679' 'Dnah14' 'Mark1' 'Hspa5'
 'Ubr1' 'Nkrf' 'Hprt' 'Mamld1' 'Xist' '2810403D21Rik' 'Mrps21' 'Meaf6'
 'Galnt9' 'Cit' 'Ncor2' 'Scarb1' 'Iqce' 'Calu' 'Mzf1' 'Xpo6' 'Sez6l2'
 'Adgra1' 'Sar1a' 'Syn3' 'Cnot2' 'Cdk4' 'Rpgrip1l' 'Nip7' 'Zmiz1' 'Ipo5'
 'Parp6' 'Gm46124' 'Pitpnm3' 'Sstr2' 'Pomc' 'Ccdc88c' 'Ccnt1' 'Nectin3'
 'Zfp760' 'Safb2' 'Uty' 'Rin1']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_52_L6_IT_CTX_Glut_4.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_52_L6_IT_CTX_Glut_4.csv
CPU times: user 58min 59s, sys: 26.6 s, total: 59min 25s
Wall time: 59min 27s


In [12]:
adata

AnnData object with n_obs × n_vars = 1718 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time', 'label', 'conf'
    var: 'feature_types'
    layers: 'norm'